In [ ]:
Oracle AI Data Platform v1.0

Copyright © 2025, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Jira Cloud Connector Samples

Read-only ingestion samples for Jira Cloud using its REST API and Python `requests`. Replace all placeholders before running a sample.

## Prerequisites

1. **Credential.** Create a **Secret Token** credential in the Credential Store with keys `site`, `email` and `token`; the token comes from https://id.atlassian.com/manage/api-tokens.
2. **Network.** The cluster needs HTTPS access to your Jira Cloud site; no library needs installing.

## Configuration

Sets the credential name, the target table and the issue query.

In [ ]:
CREDENTIAL_NAME = "<CREDENTIAL_NAME>"  # Secret Token credential with keys site, email, token
TARGET = "<CATALOG>.<SCHEMA>.<TABLE_NAME>"
JQL = "<JQL_QUERY>"  # for example: project = <PROJECT_KEY>; no ORDER BY
PAGE_SIZE = 100
OVERLAP_SECONDS = 300

# Pin the Spark session to UTC so the watermark read back from Delta is unambiguous.
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Jira Client

Helper functions for retries, paging and turning issues into rows.

In [ ]:
import math
import re
import time
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

import requests

FIELDS = [("key", "STRING"), ("summary", "STRING"), ("status", "STRING"), ("priority", "STRING"),
          ("assignee", "STRING"), ("reporter", "STRING"), ("issuetype", "STRING"),
          ("project", "STRING"), ("created", "TIMESTAMP"), ("updated", "TIMESTAMP")]
MAX_BACKOFF_SECONDS = 60.0


# Retry rate limits, 502/503/504 and network errors a bounded number of times, honouring Retry-After.
def request_json(make_request, retries=5):
    for attempt in range(retries + 1):
        try:
            response = make_request()
        except (requests.ConnectionError, requests.Timeout):
            if attempt == retries:
                raise
            time.sleep(min(2 ** attempt, MAX_BACKOFF_SECONDS))
            continue
        status = response.status_code
        if status in (429, 502, 503, 504) and attempt < retries:
            try:
                wait = float(response.headers.get("Retry-After"))
            except (TypeError, ValueError):
                wait = float(2 ** attempt)
            if not math.isfinite(wait):
                wait = float(2 ** attempt)
            time.sleep(min(max(wait, 0.0), MAX_BACKOFF_SECONDS))
            continue
        if status in (401, 403):
            raise PermissionError("HTTP {}: check the Jira email and API token".format(status))
        if status >= 400:
            raise RuntimeError("HTTP {} from Jira: {}".format(status, response.text[:300]))
        return response.json()


def account_timezone():
    url = "https://{}/rest/api/3/myself".format(site)
    return request_json(lambda: session.get(url, timeout=60))["timeZone"]


def jql_time(value, tz):
    return value.astimezone(tz).strftime("%Y-%m-%d %H:%M")


# Jira reads date-times in a query in the account's own timezone, so the bounds are written in it.
# Results are paged with the token Jira returns; a repeated token stops the loop.
def search_issues(query, since=None, until=None):
    if re.search(r"order\s+by", query, re.I):
        raise ValueError("JQL must not contain ORDER BY")
    tz = ZoneInfo(TZ_NAME)
    parts = ["({})".format(query.strip())]
    if since is not None:
        parts.append('updated >= "{}"'.format(jql_time(since - timedelta(seconds=OVERLAP_SECONDS), tz)))
    if until is not None:
        parts.append('updated <= "{}"'.format(jql_time(until, tz)))
    body = {"jql": " AND ".join(parts) + " ORDER BY updated ASC, key ASC",
            "fields": [name for name, _ in FIELDS if name != "key"], "maxResults": PAGE_SIZE}
    url = "https://{}/rest/api/3/search/jql".format(site)
    seen = set()
    while True:
        page = request_json(lambda: session.post(url, json=body, timeout=60))
        yield from page.get("issues", [])
        if page.get("isLast"):
            return
        token = page.get("nextPageToken")
        if not token or token in seen:
            raise RuntimeError("Jira paging did not advance")
        seen.add(token)
        body["nextPageToken"] = token


def to_row(issue):
    fields = issue.get("fields") or {}
    row = [issue["key"]]
    for name, kind in FIELDS[1:]:
        value = fields.get(name)
        if value is None:
            row.append(None)
        elif name in ("status", "priority", "issuetype"):
            row.append(value.get("name"))
        elif name in ("assignee", "reporter"):
            row.append(value.get("displayName"))
        elif name == "project":
            row.append(value.get("key"))
        elif kind == "TIMESTAMP":
            row.append(datetime.strptime(value, "%Y-%m-%dT%H:%M:%S.%f%z"))
        else:
            row.append(value)
    return tuple(row)


# The overlap can return an issue twice; keep one row per key, the newest.
def to_dataframe(issues):
    oldest = datetime.min.replace(tzinfo=timezone.utc)
    newest = {}
    for row in map(to_row, issues):
        if row[0] not in newest or (row[-1] or oldest) >= (newest[row[0]][-1] or oldest):
            newest[row[0]] = row
    schema = ", ".join("{} {}".format(name, kind) for name, kind in FIELDS)
    return spark.createDataFrame(list(newest.values()), schema)

## Connect to Jira

Reads the credential from the Credential Store and checks access with a single request.

In [ ]:
def credential(key):
    return aidputils.secrets.get(name=CREDENTIAL_NAME, key=key).strip()


# The API token is sent to `site`, so only a Jira Cloud host is accepted.
site = re.sub(r"^https?://", "", credential("site"), flags=re.I).rstrip("/").lower()
if not re.fullmatch(r"[a-z0-9][a-z0-9-]*\.atlassian\.net", site):
    raise ValueError("site must be a Jira Cloud host such as <site>.atlassian.net")
session = requests.Session()
session.auth = (credential("email"), credential("token"))
session.headers["Accept"] = "application/json"

# One request: proves network access and the credentials, and gives the account timezone.
TZ_NAME = account_timezone()
print("connected; account timezone:", TZ_NAME)

## Ingestion Sample

Reads every issue that matches `JQL` into a DataFrame.

In [ ]:
jira_df = to_dataframe(search_issues(JQL))

jira_df.show()

## Incremental Load into Delta

Loads every matching issue on the first run, then only issues updated since the last run, merged on `key`.

In [ ]:
# Read the watermark as a UTC string: a collected TIMESTAMP comes back in the Python
# process's local timezone. Later runs re-read OVERLAP_SECONDS before it; the MERGE removes
# the duplicates. Bounds are rounded to the minute, so an edit made less than a minute
# before a run is picked up by the next run. A watermark does not see deleted issues.
since = None
if spark.catalog.tableExists(TARGET):
    newest = spark.sql(
        f"SELECT date_format(max(updated), 'yyyy-MM-dd HH:mm:ss.SSS') AS m FROM {TARGET}"
    ).first()["m"]
    if newest is not None:
        since = datetime.strptime(newest, "%Y-%m-%d %H:%M:%S.%f").replace(tzinfo=timezone.utc)

until = datetime.now(timezone.utc)  # fixed for this run
jira_df_incremental = to_dataframe(search_issues(JQL, since, until))
print("issues read:", jira_df_incremental.count())

if not spark.catalog.tableExists(TARGET):
    jira_df_incremental.write.format("delta").saveAsTable(TARGET)
else:
    jira_df_incremental.createOrReplaceTempView("incoming_issue")
    spark.sql(
        f"MERGE INTO {TARGET} t USING incoming_issue s ON t.key = s.key "
        "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *"
    )

print("issues in target:", spark.table(TARGET).count())

## Connector Options

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `CREDENTIAL_NAME` | Credential Store entry name | Y | Secret Token credential with keys `site`, `email` and `token`. |
| `TARGET` | Three-part Delta table name | Y | Table the incremental load writes to; its schema must exist. |
| `JQL` | Jira query without an ORDER BY clause | Y | Selects the issues, for example `project = <PROJECT_KEY>`. |
| `PAGE_SIZE` | Integer from 1 to 5000 | N | Issues per search request. Default 100. |
| `OVERLAP_SECONDS` | Non-negative integer | N | How far before the watermark later runs re-read. Default 300. |